In [66]:
using Pkg, JuMP, Gurobi, Random

Pour le sous problème sans PL, tester d'abord si sum(y_i) = d
Gurobi n'a pas de Benders intégré (utiliser CPLEX sous Windows)

### Exercice 1

In [2]:
function loadData(n::Int)
    d = div(n,2)
    f = zeros(Int, n)
    c = zeros(Int, n)
    f[1] = 7
    c[1] = 8
    for i in 2:n
        f[i] = rem(f[i-1]*f[1], 159)
        c[i] = rem(c[i-1]*c[1], 61)
    end
    return n, d, f, c
end

function loadBaseData()
    n = 5
    f = [7, 2, 2, 7, 7]
    c = [666, 5, 4, 3, 2]
    d = 2
    return n, d, f, c
end

loadBaseData (generic function with 1 method)

In [ ]:
function compactModel(n::Int, d::Int, f::Vector{Int64}, c::Vector{Int64}, TimeLimit::Int=60)

    mod = Model(Gurobi.Optimizer)

    set_optimizer_attribute(mod, "OutputFlag", 0)
    set_optimizer_attribute(mod, "TimeLimit", TimeLimit)

    @variable(mod, y[1:n], Bin)
    @variable(mod, x[1:n] >= 0)

    @constraint(mod, sum(x[i] for i in 1:n) == d)
    @constraint(mod, [i in 1:n], x[i] <= y[i])

    # Contraintes de design
    @constraint(mod, y[1] >= y[2])
    @constraint(mod, y[1] >= y[3])

    @objective(mod, Min, sum(f[i] * y[i] + c[i] * x[i] for i in 1:n))

    optimize!(mod)

    optimum = JuMP.objective_value(mod)
    lb = MOI.get(mod, MOI.ObjectiveBound())
    gap = MOI.get(mod, MOI.RelativeGap())
    solve_time = MOI.get(mod, MOI.SolveTimeSec())
    nodes = MOI.get(mod, MOI.NodeCount())
    y_opt = JuMP.value(y)
    x_opt = JuMP.value(x)

    println("Found an optimal solution of $optimum in $(round(solve_time, digits=3)) seconds and $nodes nodes")
    println("Opened $(sum(y_opt[i] for i in 1:n)) sites")

    return (optimum, y_opt, x_opt)
end

In [27]:
function separation_PL(n::Int, d::Int, c::Vector{Int64}, y::Vector{})

    println("Starting separation")

    mod = Model(Gurobi.Optimizer)

    set_optimizer_attribute(mod, "OutputFlag", 0)
    set_optimizer_attribute(mod, "TimeLimit", 60)

    @variable(mod, v[1:n] >= 0)
    @variable(mod, b)

    @constraint(mod, [i in 1:n], b - v[i] <= c[i])

    @objective(mod, Max, d * b - sum(y[i] * v[i] for i in 1:n))

    optimize!(mod)

    println(termination_status(mod))

    optimum = JuMP.objective_value(mod) 
    v_opt = JuMP.value(v)
    b_opt = JuMP.value(b)

    return (optimum, v_opt, b_opt)

end

separation_PL (generic function with 1 method)

In [46]:
function separation_speedy(n::Int, d::Int, c::Vector{Int64}, y::Vector{})

    b_opt = maximum(c[i] for i in 1:n)

    v_opt = zeros(n)
    for i in 1:n
        v_opt[i] = b_opt - c[i]
    end

    optimum = d * b_opt - sum(y[i] * v_opt[i] for i in 1:n)

    return (optimum, v_opt, b_opt)
end

separation_speedy (generic function with 1 method)

In [47]:
function main(n::Int, d::Int, f::Vector{Int}, c::Vector{Int}, B::Vector{}, V::Vector{})

    println("Starting main")
    
    mod = Model(Gurobi.Optimizer)

    set_optimizer_attribute(mod, "OutputFlag", 0)
    set_optimizer_attribute(mod, "TimeLimit", 60)

    @variable(mod, y[1:n], Bin)
    @variable(mod, w >= 0)

    # Contraintes de design
    @constraint(mod, y[1] >= y[2])
    @constraint(mod, y[1] >= y[3])

    # Coupe de faisabilité
    @constraint(mod, sum(y[i] for i in 1:n) >= d)

    # Coupes d'optimalité
    @constraint(mod, [l in eachindex(B)], w >= d * B[l] - sum(y[i] * V[l][i] for i in 1:n))

    @objective(mod, Min, sum(f[i] * y[i] for i in 1:n) + w)

    optimize!(mod)

    optimum = JuMP.objective_value(mod)
    lb = MOI.get(mod, MOI.ObjectiveBound())
    gap = MOI.get(mod, MOI.RelativeGap())
    solve_time = MOI.get(mod, MOI.SolveTimeSec())
    nodes = MOI.get(mod, MOI.NodeCount())
    y_opt = JuMP.value(y)
    w_opt = JuMP.value(w)

    return (optimum, y_opt, w_opt)

end



main (generic function with 1 method)

In [ ]:
function benders(n::Int, d::Int, f::Vector{Int}, c::Vector{Int}, speedy_sep::Bool=false)

    # Initialize values
    optimum = 0
    y_opt = zeros(n)
    B = []
    V = []
    optimal = false
    t0 = time_ns()
    
    while (!optimal)

        optimal = true

        (optimum, y_opt, w_opt) = main(n, d, f, c, B, V)

        if (speedy_sep) && (sum(y_opt[i] for i in 1:n) == d)
            println("Doing Speedy Sep, sum(y) = $(sum(y_opt[i] for i in 1:n))")
            (sep_optimum, v_opt, b_opt) = separation_speedy(n, d, c, y_opt)
        else
            println("Doing PL Sep, sum(y) = $(sum(y_opt[i] for i in 1:n))")
            (sep_optimum, v_opt, b_opt) = separation_PL(n, d, c, y_opt)
        end

        if (w_opt < sep_optimum)
            optimal = false
            push!(V, copy(v_opt))
            push!(B, copy(b_opt))
        end

    end

    println("L'optimum vaut $(optimum)")
    elapsed = (time_ns() - t0) / 1e9

    return (optimum, y_opt, elapsed)
end

benders (generic function with 2 methods)

In [ ]:
(n, d, f, c) = loadData(10000)
benders(n, d, f, c, true)

Starting main
Set parameter Username
Set parameter LicenseID to value 2780083
Academic license - for non-commercial use only - expires 2027-02-17
Set parameter TimeLimit to value 60
Doing Speedy Sep, sum(y) = 500000.0
Starting main
Set parameter Username
Set parameter LicenseID to value 2780083
Academic license - for non-commercial use only - expires 2027-02-17


LoadError: InterruptException:

In [49]:
(n, d, f, c) = loadBaseData()

(5, 2, [7, 2, 2, 7, 7], [666, 5, 4, 3, 2])

In [ ]:
function compactModel(n::Int, d::Int, f::Vector{Int64}, c::Vector{Int64}, TimeLimit::Int=60, implications::Vector{} = [])

    mod = Model(Gurobi.Optimizer)

    set_optimizer_attribute(mod, "OutputFlag", 0)
    set_optimizer_attribute(mod, "TimeLimit", TimeLimit)

    @variable(mod, y[1:n], Bin)
    @variable(mod, x[1:n] >= 0)

    @constraint(mod, sum(x[i] for i in 1:n) == d)
    @constraint(mod, [i in 1:n], x[i] <= y[i])

    # Contraintes de design
    @constraint(mod, y[1] >= y[2])
    @constraint(mod, y[1] >= y[3])

    @objective(mod, Min, sum(f[i] * y[i] + c[i] * x[i] for i in 1:n))

    optimize!(mod)

    optimum = JuMP.objective_value(mod)
    lb = MOI.get(mod, MOI.ObjectiveBound())
    gap = MOI.get(mod, MOI.RelativeGap())
    solve_time = MOI.get(mod, MOI.SolveTimeSec())
    nodes = MOI.get(mod, MOI.NodeCount())
    y_opt = JuMP.value(y)
    x_opt = JuMP.value(x)

    println("Found an optimal solution of $optimum in $(round(solve_time, digits=3)) seconds and $nodes nodes")
    println("Opened $(sum(y_opt[i] for i in 1:n)) sites")

    return (optimum, y_opt, x_opt)
end

compactModel (generic function with 6 methods)

In [61]:
(optimum, y_opt, x_opt) = compactModel(n,d,f,c)

Set parameter Username
Set parameter LicenseID to value 2780083
Academic license - for non-commercial use only - expires 2027-02-17
Set parameter TimeLimit to value 60
Found an optimal solution of 19.0 in 0.002 seconds and 1 nodes
Opened 2.0 sites


(19.0, [-0.0, 0.0, 0.0, 1.0, 1.0], [0.0, 0.0, 0.0, 1.0, 1.0])